In [34]:
from bs4 import BeautifulSoup
import requests
import time
from PIL import Image, ImageDraw, ImageFont

In [35]:
urls = {"TAMIL": "https://ta.wikipedia.org/",
        "MALAYALAM": "https://ml.wikipedia.org/",
        "KANNADA": "https://kn.wikipedia.org/",
        "TELUGU": "https://te.wikipedia.org/",
       }

In [36]:
custom_headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept': 'application/json',
}


In [37]:
import unicodedata
import string

In [38]:
def clean_text(text, language):
    exceptions = list(map(unicodedata.name, [".", ",", "(", ")", "+", "-"]))
    remove = "•\xa0"
    result = ""
    for c in text:
        if c in set(string.whitespace + remove):
            if len(result) != 0 and result[len(result)-1] != " ":
                result+=" "
            continue
    
        if unicodedata.name(c) in exceptions or unicodedata.name(c).split()[0] == "DIGIT" or unicodedata.name(c).split()[0] == language:   
            result+=c
    return result

In [39]:
fonts = {
    "TAMIL": "fonts/Noto_Sans_Tamil/static/NotoSansTamil-Regular.ttf",
    "MALAYALAM": "fonts/Noto_Sans_Malayalam/static/NotoSansMalayalam-Regular.ttf",
    "TELUGU": "fonts/Noto_Sans_Telugu/static/NotoSansTelugu-Regular.ttf",
    "KANNADA": "fonts/Noto_Sans_Kannada/static/NotoSansKannada-Regular.ttf",
}
import random

In [40]:
def text_to_image(text, language):
    font = ImageFont.truetype(
        fonts[language],
        # 20
        random.randint(25, 40)
    )
    padding = 20
    max_width = 1000 
    
    temp = Image.new("RGB", (1, 1))
    draw = ImageDraw.Draw(temp)
    
    lines = []
    line = ""
    
    for word in text.split():
        test = f"{line} {word}".strip()
    
        if draw.textlength(test, font=font) <= max_width:
            line = test
        else:
            if line:
                lines.append(line)
            line = word
    
    if line:
        lines.append(line)
    
    wrapped_text = "\n".join(lines)
    
    bbox = draw.multiline_textbbox(
        (0, 0), wrapped_text, font=font, spacing=8
    )
    # print(bbox)
    width = max_width + padding * 2
    height = bbox[3] - bbox[1] + padding * 2
    
    # Create image and draw text
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    
    draw.multiline_text(
        (padding, padding - bbox[1]),
        wrapped_text,
        font=font,
        fill="black",
        spacing=15
    )
    return img



In [41]:
import os
for language, url in urls.items():
    text = ""
    max_len = 1_000_000
    while len(text) < max_len:
        random_url = url + "wiki/Special:Random"
        response = requests.get(url, headers=custom_headers)
        if response.status_code != 200: 
            print("non 200 status returned", response.text)
            print("waiting 5 seconds")
            time.sleep(5)
            continue
        beautifulsoup = BeautifulSoup(response.text)
        text += clean_text(beautifulsoup.text, language)+ ' '
    text = text[:max_len]

    j=1
    os.makedirs(f"data/{language}", exist_ok=True)
    for i in range(0, max_len, 1000):
        chunk = text[i:i + 1000]
        img = text_to_image(chunk, language)
        img.save(f"data/{language}/{language}-{j}.png")
        j+=1
